# 📊 04 — Model Evaluation & GradCAM
**Project:** Diabetic Retinopathy Detection

**Goal:** Evaluate the trained model on the held-out test set. Generate confusion matrix, per-class metrics, ROC curves, and GradCAM visualizations.

---

## 📦 1. Imports & Load Model

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    confusion_matrix, classification_report,
    cohen_kappa_score, roc_auc_score, roc_curve
)
from pathlib import Path

IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_CLASSES = 5
DATA_DIR    = Path('../data/processed')
MODEL_DIR   = Path('../models')
FIG_DIR     = Path('../results/figures')

LABEL_MAP  = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative'}
CLASS_NAMES = [LABEL_MAP[i] for i in range(5)]

# Load best model
model = keras.models.load_model(MODEL_DIR / 'best_model.keras')
print('Model loaded successfully.')
print(f'Total parameters: {model.count_params():,}')

## 🔮 2. Generate Predictions on Test Set

In [ ]:
test_df = pd.read_csv(DATA_DIR / 'test.csv')

def preprocess_image(path):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    return (img - mean) / std

images = np.array([preprocess_image(p) for p in test_df['image_path']])
y_true = test_df['diagnosis'].values

# Predict in batches
y_prob = model.predict(images, batch_size=BATCH_SIZE, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print(f'\nTest set size: {len(y_true):,}')
print(f'Predictions shape: {y_prob.shape}')

## 📐 3. Core Metrics

In [ ]:
accuracy = (y_pred == y_true).mean()
qwk      = cohen_kappa_score(y_true, y_pred, weights='quadratic')

# OvR AUC
y_true_onehot = np.eye(NUM_CLASSES)[y_true]
macro_auc = roc_auc_score(y_true_onehot, y_prob, multi_class='ovr', average='macro')

print('=' * 45)
print(f'  Test Accuracy            : {accuracy:.4f} ({accuracy*100:.1f}%)')
print(f'  Quadratic Weighted Kappa : {qwk:.4f}')
print(f'  Macro AUC (OvR)          : {macro_auc:.4f}')
print('=' * 45)

print('\nPer-class classification report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

## 🔲 4. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalized

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, data, title, fmt in zip(
    axes,
    [cm, cm_norm],
    ['Confusion Matrix (Counts)', 'Confusion Matrix (Row-Normalized)'],
    ['d', '.2f']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.5, linecolor='white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle(f'EfficientNetB3 Test Performance  |  Accuracy: {accuracy:.1%}  |  QWK: {qwk:.3f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 📉 5. ROC Curves (One-vs-Rest)

In [ ]:
colors = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
plt.figure(figsize=(9, 7))

for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_true_onehot[:, i], y_prob[:, i])
    auc = roc_auc_score(y_true_onehot[:, i], y_prob[:, i])
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{cls_name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random (AUC = 0.500)')
plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.02])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — One-vs-Rest per DR Grade', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔥 6. GradCAM — Model Explainability

In [ ]:
def get_gradcam_heatmap(model, img_array, layer_name='top_conv'):
    """Generate GradCAM heatmap for a given image."""
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array[np.newaxis], training=False)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)


def overlay_gradcam(original_img, heatmap, alpha=0.4):
    heatmap = cv2.resize(heatmap, (original_img.shape[1], original_img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    superimposed = cv2.addWeighted(original_img, 1 - alpha, heatmap_color, alpha, 0)
    return superimposed


# Plot GradCAM for one sample per class
fig, axes = plt.subplots(5, 3, figsize=(14, 22))
col_titles = ['Original', 'GradCAM Heatmap', 'Overlay']

for grade in range(5):
    sample = test_df[test_df['diagnosis'] == grade].sample(1, random_state=42)
    path   = sample['image_path'].values[0]

    raw = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    raw = cv2.resize(raw, (IMG_SIZE, IMG_SIZE))

    img_norm = raw.astype(np.float32) / 255.0
    img_norm = (img_norm - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])

    try:
        heatmap, pred_class = get_gradcam_heatmap(model, img_norm)
        overlay = overlay_gradcam(raw, heatmap)
    except Exception:
        heatmap = np.zeros((7, 7))
        pred_class = grade
        overlay = raw.copy()

    axes[grade, 0].imshow(raw)
    axes[grade, 0].set_ylabel(f'Grade {grade}\n{LABEL_MAP[grade]}', fontsize=10, fontweight='bold',
                               rotation=0, labelpad=80, va='center')
    axes[grade, 1].imshow(heatmap, cmap='jet')
    axes[grade, 2].imshow(overlay)
    axes[grade, 2].set_title(f'Pred: {LABEL_MAP[pred_class]}', fontsize=9,
                              color='green' if pred_class == grade else 'red')

    for col in range(3):
        axes[grade, col].axis('off')
        if grade == 0:
            axes[grade, col].set_title(col_titles[col], fontsize=12, fontweight='bold')

plt.suptitle('GradCAM Visualizations — What the Model Focuses On', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'gradcam_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## ✅ 7. Final Results Summary

In [ ]:
print('=' * 50)
print('  FINAL RESULTS — Diabetic Retinopathy Detection')
print('=' * 50)
print(f'  Model          : EfficientNetB3 + Custom Head')
print(f'  Test Accuracy  : {accuracy:.4f} ({accuracy*100:.1f}%)')
print(f'  QWK            : {qwk:.4f}')
print(f'  Macro AUC      : {macro_auc:.4f}')
print('=' * 50)
print('\nSaved artifacts:')
for f in FIG_DIR.glob('*.png'):
    print(f'  ✓ {f.name}')